# Deep Crossentropy method

In this section we'll extend your CEM implementation with neural networks! You will train a multi-layer neural network to solve simple continuous state space games. __Please make sure you're done with tabular crossentropy method from the seminar notebook.__

![img](https://watanimg.elwatannews.com/old_news_images/large/249765_Large_20140709045740_11.jpg)



In [ ]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/setup_colab.sh -O- | bash
    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

In [ ]:
# Install gymnasium if you didn't
!pip install gymnasium[toy_text,classic_control]

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# if you see "<classname> has no attribute .env", remove .env or update gym
env = gym.make("CartPole-v1", render_mode="rgb_array").env

env.reset(seed=42)
n_actions = env.action_space.n
state_dim = env.observation_space.shape[0]

plt.imshow(env.render())
print("state vector dim =", state_dim)
print("n_actions =", n_actions)

env.close()

In [ ]:
env.reset()[0]

# Neural Network Policy

For this assignment we'll utilize the simplified neural network implementation from __[Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)__. Here's what you'll need:

* `agent.partial_fit(states, actions)` - make a single training pass over the data. Maximize the probability of :actions: from :states:
* `agent.predict_proba(states)` - predict probabilities of all actions, a matrix of shape __[len(states), n_actions]__


In [ ]:
list(range(n_actions))

In [ ]:
from sklearn.neural_network import MLPClassifier

agent = MLPClassifier(
    hidden_layer_sizes=(512, 512),
    activation="relu",
)

# initialize agent to the dimension of state space and number of actions
agent.partial_fit([env.reset()[0]] * n_actions, range(n_actions), classes=range(n_actions))


In [ ]:
s, _ = env.reset()
agent.predict_proba([s])[0]

In [ ]:
def generate_session(env, agent, t_max=1000):
    """
    Play a single game using agent neural network.
    Terminate when game finishes or after :t_max: steps
    """
    states, actions = [], []
    total_reward = 0

    s, _ = env.reset()

    for t in range(t_max):

        # use agent to predict a vector of action probabilities for state :s:
        probs = agent.predict_proba([s])[0]

        assert probs.shape == (env.action_space.n,), "make sure probabilities are a vector (hint: np.reshape)"

        # use the probabilities you predicted to pick an action
        # sample proportionally to the probabilities, don't just take the most likely action
        a = np.random.choice(n_actions, p=probs)
        # ^-- hint: try np.random.choice

        new_s, r, terminated, truncated, _ = env.step(a)

        # record sessions like you did before
        states.append(s)
        actions.append(a)
        total_reward += r

        s = new_s
        if terminated or truncated:
            break
    return states, actions, total_reward


In [ ]:
dummy_states, dummy_actions, dummy_reward = generate_session(env, agent, t_max=5)
print("states:", np.stack(dummy_states))
print("actions:", dummy_actions)
print("reward:", dummy_reward)


### CEM steps
Deep CEM uses exactly the same strategy as the regular CEM, so you can copy your function code from previous notebook.

The only difference is that now each observation is not a number but a `float32` vector.

In [ ]:
def select_elites(states_batch, actions_batch, rewards_batch, percentile=50):
    """
    Select states and actions from games that have rewards >= percentile
    :param states_batch: list of lists of states, states_batch[session_i][t]
    :param actions_batch: list of lists of actions, actions_batch[session_i][t]
    :param rewards_batch: list of rewards, rewards_batch[session_i]

    :returns: elite_states,elite_actions, both 1D lists of states and respective actions from elite sessions

    Please return elite states and actions in their original order
    [i.e. sorted by session number and timestep within session]

    If you are confused, see examples below. Please don't assume that states are integers
    (they will become different later).
    """

    elite_indices = np.where(rewards_batch >= np.percentile(rewards_batch, percentile))[0]
    elite_states = []
    elite_actions = []
    for i in elite_indices:
        elite_states.extend(states_batch[i])
        elite_actions.extend(actions_batch[i])

    return elite_states, elite_actions


# Training loop
Generate sessions, select N best and fit to those.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display


def show_progress(rewards_batch, log, percentile, reward_range=[-990, +10]):
    mean_reward = np.mean(rewards_batch)
    threshold = np.percentile(rewards_batch, percentile)
    log.append([mean_reward, threshold])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=[8, 4])
    fig.suptitle("mean reward = %.3f, threshold = %.3f" % (mean_reward, threshold))

    ax1.plot(list(zip(*log))[0], label="Mean rewards")
    ax1.plot(list(zip(*log))[1], label="Reward thresholds")
    ax1.legend()
    ax1.grid()

    ax2.hist(rewards_batch, range=reward_range)
    ax2.vlines([threshold], [0], [100], label="percentile", color="red")
    ax2.legend()
    ax2.grid()

    handle = getattr(show_progress, "_handle", None)
    if handle is None:
        show_progress._handle = display(fig, display_id=True)
    else:
        handle.update(fig)

    plt.close(fig)

In [ ]:
n_sessions = 1000
percentile = 85
log = []

for i in range(1000):
    # generate new sessions
    sessions = [generate_session(env=env, agent=agent) for _ in range(n_actions)]

    states_batch, actions_batch, rewards_batch = zip(*sessions)

    elite_states, elite_actions = select_elites(states_batch, actions_batch, rewards_batch, percentile)

    # <YOUR CODE: partial_fit agent to predict elite_actions(y) from elite_states(X)>
    agent = agent.partial_fit(elite_states, elite_actions, classes=range(n_actions))

    show_progress(
        rewards_batch, log, percentile, reward_range=[0, np.max(rewards_batch)]
    )

    if np.mean(rewards_batch) > 190:
        print("You Win! You may stop training now via KeyboardInterrupt.")



### Student comment

Честно, сказать не получилось сделать что то с количеством параметров типа $32\times32$, что еще хоть как то похоже на обучение
С таким количеством параметров как у меня $(512\times512)$, можно в целом заучить просто всю выборку 

# Results

In [ ]:
# Record sessions

from gymnasium.wrappers import RecordVideo

with RecordVideo(
    env=gym.make("CartPole-v1", render_mode="rgb_array"),
    video_folder="./videos",
    episode_trigger=lambda episode_number: True,
) as env_monitor:
    sessions = [generate_session(env_monitor, agent) for _ in range(100)]


In [ ]:
# Show video. This may not work in some setups. If it doesn't
# work for you, you can download the videos and view them locally.

from pathlib import Path
from base64 import b64encode
from IPython.display import HTML

video_paths = sorted([s for s in Path("videos").iterdir() if s.suffix == ".mp4"])
video_path = video_paths[-1]  # You can also try other indices

if "google.colab" in sys.modules:
    # https://stackoverflow.com/a/57378660/1214547
    with video_path.open("rb") as fp:
        mp4 = fp.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
else:
    data_url = str(video_path)

HTML(
    """
<video width="640" height="480" controls>
  <source src="{}" type="video/mp4">
</video>
""".format(
        data_url
    )
)


# Homework part I

### Tabular crossentropy method

You may have noticed that the taxi problem quickly converges from -100 to a near-optimal score and then descends back into -50/-100. This is in part because the environment has some innate randomness. Namely, the starting points of passenger/driver change from episode to episode.

### Tasks
- __1.1__ (0.1 pts) Find out how the algorithm performance changes if you use a different `percentile` and/or `n_sessions`. Provide here some figures so we can see how the hyperparameters influence the performance.
- __1.2__ (0.1 pts) Tune the algorithm to end up with positive average score.

It's okay to modify the existing code.


```<Describe what you did here>```

Весь код в одной ячейке пункты 1.1 и 1.2 далее

In [ ]:
import sys
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

print(f"python {sys.version.split()[0]} | gymnasium {gym.__version__} | numpy {np.__version__}")

SEED = 1
np.random.seed(SEED)  # чтобы графики воспроизводились между запусками

env = gym.make("Taxi-v4", render_mode="rgb_array")
obs, info = env.reset(seed=SEED)
print("start state:", obs, "| info:", info)

plt.imshow(env.render())
plt.axis("off");

n_states = env.observation_space.n
n_actions = env.action_space.n

print(f"n_states={n_states}, n_actions={n_actions}")

def initialize_policy(n_states, n_actions):
    policy = np.ones((n_states, n_actions)) / n_actions

    return policy

policy = initialize_policy(n_states, n_actions)
assert type(policy) in (np.ndarray, np.matrix)
assert np.allclose(policy, 1.0 / n_actions)
assert np.allclose(np.sum(policy, axis=1), 1)

def generate_session(env, policy, t_max=10**4):
    """
    Play game until end or for t_max ticks.
    :param policy: an array of shape [n_states,n_actions] with action probabilities
    :returns: list of states, list of actions and sum of rewards
    """
    states, actions = [], []
    total_reward = 0.0

    s, _ = env.reset()

    for t in range(t_max):
        # Hint: you can use np.random.choice for sampling action
        # https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html

        #<YOUR CODE: sample action from policy>
        a = np.random.choice(n_actions, p=policy[s])

        new_s, r, terminated, truncated, _ = env.step(a)

        # Record information we just got from the environment.
        states.append(s)
        actions.append(a)
        total_reward += r

        s = new_s
        if terminated or truncated:
            break

    return states, actions, total_reward

s, a, r = generate_session(env, policy)
assert type(s) == type(a) == list
assert len(s) == len(a)
assert type(r) in [float, np.float64]

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display


def show_progress(rewards_batch, log, percentile, policy=None, elapsed=None,
                  reward_range=(-990, 10), plot=True):
    """
    A convenience function that displays training progress.
    No cool math here, just charts.
    """

    mean_reward = np.mean(rewards_batch)
    threshold = np.percentile(rewards_batch, percentile)
    solved = np.mean(np.asarray(rewards_batch) > 0)  # доля эпизодов с положительной наградой

    # энтропия стратегии: показывает, насколько стратегия ещё исследует,
    # а не схлопнулась в детерминизм
    if policy is None:
        entropy = np.nan
    else:
        entropy = -(policy * np.log(policy + 1e-12)).sum(axis=1).mean()

    log.append([mean_reward, threshold, solved, entropy])

    if not plot:
        return

    mean_rewards, thresholds, solved_rates, entropies = (list(x) for x in zip(*log))

    fig, (ax1, ax2, ax4) = plt.subplots(1, 3, figsize=(13, 4))

    tail = f" | {elapsed:.1f} s/iter" if elapsed is not None else ""
    fig.suptitle(
        f"iter {len(log)} | mean = {mean_reward:.1f} | threshold = {threshold:.1f}"
        f" | solved = {solved:.0%}" + tail
    )

    ax1.plot(mean_rewards, label="Mean rewards")
    ax1.plot(thresholds, label="Reward thresholds")
    ax1.axhline(7.9, color="green", ls="--", lw=1, label="оптимум (~7.9)")
    ax1.axhline(0, color="gray", ls=":", lw=1)
    ax1.legend()
    ax1.grid()

    ax2.plot(solved_rates, color="tab:orange", label="Доля успешных эпизодов")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid()
    handles = ax2.get_lines()
    if not np.all(np.isnan(entropies)):
        ax3 = ax2.twinx()
        ax3.plot(entropies, color="tab:purple", label="Энтропия стратегии")
        ax3.set_ylim(bottom=0)
        handles += ax3.get_lines()
    ax2.legend(handles, [h.get_label() for h in handles], loc="upper left")

    ax4.hist(rewards_batch, range=reward_range, bins=20)
    ax4.axvline(threshold, color="red", label="percentile")
    ax4.set_ylim(0, len(rewards_batch))
    ax4.legend()
    ax4.grid()

    fig.tight_layout()

    if len(log) == 1:              # новый прогон — новый вывод
        show_progress._handle = display(fig, display_id=True)
    else:
        show_progress._handle.update(fig)

    plt.close(fig)

def select_elites(states_batch, actions_batch, rewards_batch, percentile):
    """
    Select states and actions from games that have rewards >= percentile
    :param states_batch: list of lists of states, states_batch[session_i][t]
    :param actions_batch: list of lists of actions, actions_batch[session_i][t]
    :param rewards_batch: list of rewards, rewards_batch[session_i]

    :returns: elite_states,elite_actions, both 1D lists of states and respective actions from elite sessions

    Please return elite states and actions in their original order
    [i.e. sorted by session number and timestep within session]

    If you are confused, see examples below. Please don't assume that states are integers
    (they will become different later).
    """

    reward_threshold = np.percentile(rewards_batch, percentile)
    (elites, ) = np.where(rewards_batch >= reward_threshold)
    
    elite_states = []
    elite_actions = []
    for i in elites:
        elite_states.extend(states_batch[i])
        elite_actions.extend(actions_batch[i])

    return elite_states, elite_actions

def get_new_policy(elite_states, elite_actions):
    """
    Given a list of elite states/actions from select_elites,
    return a new policy where each action probability is proportional to

        policy[s_i,a_i] ~ #[occurrences of s_i and a_i in elite states/actions]

    Don't forget to normalize the policy to get valid probabilities and handle the 0/0 case.
    For states that you never visited, use a uniform distribution (1/n_actions for all states).

    :param elite_states: 1D list of states from elite sessions
    :param elite_actions: 1D list of actions from elite sessions

    """

    new_policy = np.zeros([n_states, n_actions])

    #<YOUR CODE: set probabilities for actions given elite states & actions>
    # Don't forget to set 1/n_actions for all actions in unvisited states.
    for s, a in zip(elite_states, elite_actions):
        new_policy[s, a] += 1

    new_policy[np.where(new_policy.sum(axis=1) == 0)] = 1
    new_policy = new_policy / new_policy.sum(axis=1, keepdims=True)

    return new_policy

from time import perf_counter


def train_cem(env, n_sessions=250, percentile=25, learning_rate=0.5,
              n_iterations=100, verbose=True):
    """Обучить табличную стратегию кросс-энтропийным методом.

    Args:
        env: среда gymnasium.
        n_sessions: сколько эпизодов играем на каждой итерации.
        percentile: какой процент худших эпизодов отбрасываем.
        learning_rate: скорость обновления стратегии, от 0 до 1.
        n_iterations: максимальное число итераций.
        verbose: рисовать ли графики прогресса.

    Returns:
        Пара (log, policy): история обучения и итоговая стратегия.
    """
    policy = initialize_policy(n_states, n_actions)
    log = []

    for i in range(n_iterations):
        t0 = perf_counter()

        sessions = [generate_session(env, policy) for _ in range(n_sessions)]
        states_batch, actions_batch, rewards_batch = zip(*sessions)

        elite_states, elite_actions = select_elites(
            states_batch, actions_batch, rewards_batch, percentile
        )
        new_policy = get_new_policy(elite_states, elite_actions)
        policy = learning_rate * new_policy + (1 - learning_rate) * policy

        elapsed = perf_counter() - t0
       
        show_progress(rewards_batch, log, percentile, policy=policy,
                      elapsed=elapsed, plot=verbose)

        if np.mean(rewards_batch) > 0:
            print(f"Решено за {i + 1} итераций!")
            break

    return log, policy

1.1

In [ ]:
import numpy as np
import gymnasium as gym
from joblib import Parallel, delayed


def run_cem(seed, n_sessions, percentile, lr, episode_budget=15_000):
    np.random.seed(seed)
    env = gym.make("Taxi-v")
    env.reset(seed=seed)
    policy = initialize_policy(n_states, n_actions)

    rec = []
    for i in range(episode_budget // n_sessions):
        sessions = [generate_session(env, policy) for _ in range(n_sessions)]
        s_b, a_b, r_b = zip(*sessions)
        el_s, el_a = select_elites(s_b, a_b, r_b, percentile)
        policy = lr * get_new_policy(el_s, el_a) + (1 - lr) * policy

        rec.append((
            (i + 1) * n_sessions,
            np.mean(r_b),
            np.mean(np.asarray(r_b) > 0),
            -(policy * np.log(policy + 1e-12)).sum(1).mean(),
            len(np.unique(el_s)) / n_states,
        ))

    keys = ["episodes", "mean", "solved", "entropy", "coverage"]
    return dict(zip(keys, np.asarray(rec).T))


PCTS, NSESS, SEEDS, LR, BUDGET = [25, 50, 60, 70, 80, 90], [100, 250, 500, 1000], [0, 1, 2], 0.5, 15_000

jobs = [(p, n, s) for p in PCTS for n in NSESS for s in SEEDS]
out = Parallel(n_jobs=-1, verbose=5)(delayed(run_cem)(s, n, p, LR, BUDGET) for p, n, s in jobs)

runs = {}
for (p, n, _), r in zip(jobs, out):
    runs.setdefault((p, n), []).append(r)

In [ ]:
fig, axes = plt.subplots(len(PCTS), len(NSESS), figsize=(4 * len(NSESS), 2.7 * len(PCTS)),
                         sharex=True, sharey=True)

for r, p in enumerate(PCTS):
    for c, n in enumerate(NSESS):
        ax, rs = axes[r, c], runs[(p, n)]
        ep = rs[0]["episodes"]
        curves = np.stack([x["mean"] for x in rs])

        ax.fill_between(ep, curves.min(0), curves.max(0), alpha=0.18, color="tab:blue")
        ax.plot(ep, np.median(curves, 0), color="tab:blue", lw=1.6)
        ax.axhline(0, color="gray", ls=":", lw=1)
        ax.axhline(7.9, color="green", ls="--", lw=1)
        ax.grid(alpha=0.3)
        ax.text(0.03, 0.94, f"элита ≈ {int(n * (1 - p / 100))} эп/итер",
                transform=ax.transAxes, va="top", fontsize=8, color="dimgray")

        if r == 0:
            ax.set_title(f"n_sessions = {n}")
        if c == 0:
            ax.set_ylabel(f"percentile = {p}\nmean reward")
        if r == len(PCTS) - 1:
            ax.set_xlabel("сыграно эпизодов")

fig.suptitle(f"CEM на Taxi-v4, lr={LR}, медиана и разброс по {len(SEEDS)} сидам, "
             f"равный бюджет {BUDGET} эпизодов", y=1.0)
fig.tight_layout()

In [ ]:
def tail_score(r, frac=0.1):
    k = max(1, int(len(r["mean"]) * frac))
    return r["mean"][-k:].mean()

def eps_to_solve(r):
    hit = r["mean"] > 0
    return r["episodes"][np.argmax(hit)] if hit.any() else np.nan

quality = np.array([[np.median([tail_score(r) for r in runs[(p, n)]]) for n in NSESS] for p in PCTS])
speed   = np.array([[np.nanmedian([eps_to_solve(r) for r in runs[(p, n)]]) for n in NSESS] for p in PCTS])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, M, ttl, cmap, fmt in [(axes[0], quality, "Финальный скор (хвост 10%)", "RdYlGn", "{:.0f}"),
                              (axes[1], speed, "Эпизодов до mean > 0", "viridis_r", "{:.0f}")]:
    im = ax.imshow(M, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(NSESS)), NSESS)
    ax.set_yticks(range(len(PCTS)), PCTS)
    ax.set_xlabel("n_sessions")
    ax.set_ylabel("percentile")
    ax.set_title(ttl)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            ax.text(j, i, "—" if np.isnan(v) else fmt.format(v),
                    ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax)
fig.tight_layout()

# Homework part II

### Deep crossentropy method

By this moment, you should have got enough score on [CartPole-v0](https://gymnasium.farama.org/environments/classic_control/cart_pole/) to consider it solved (see the link). It's time to try something harder.

* if you have any trouble with CartPole-v0 and feel stuck, feel free to ask us or your peers for help.

### Tasks

* __2.1__ (0.3 pts) Pick one of environments: `MountainCar-v0` or `LunarLander-v2`.
  * For MountainCar, get average reward of __at least -150__
  * For LunarLander, get average reward of __at least +50__

See the tips section below, it's kinda important.
__Note:__ If your agent is below the target score, you'll still get some of the points depending on the result, so don't be afraid to submit it.
  
  
* __2.2__ (up to 0.5 pts) Devise a way to speed up training against the default version
  * Obvious improvement: use [`joblib`](https://joblib.readthedocs.io/en/latest/). However, note that you will probably need to spawn a new environment in each of the workers instead of passing it via pickling. (0.1 pts)
  * Try re-using samples from 3-5 last iterations when computing threshold and training. (0.2 pts)
  * Obtain __-100__ at `MountainCar-v0` or __+200__ at `LunarLander-v2` (0.2 pts). Feel free to experiment with hyperparameters, architectures, schedules etc.  
  
### Tips
* Gymnasium pages: [MountainCar](https://gymnasium.farama.org/environments/classic_control/mountain_car/), [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/)
* Sessions for MountainCar may last for 10k+ ticks. Make sure ```t_max``` param is at least 10k.
 * Also it may be a good idea to cut rewards via ">" and not ">=". If 90% of your sessions get reward of -10k and 10% are better, than if you use percentile 20% as threshold, R >= threshold __fails to cut off bad sessions__ while R > threshold works alright.
* _issue with gym_: Some versions of gym limit game time by 200 ticks. This will prevent cem training in most cases. Make sure your agent is able to play for the specified __t_max__, and if it isn't, try `env = gym.make("MountainCar-v0").env` or otherwise get rid of TimeLimit wrapper.
* If you use old _swig_ lib for LunarLander-v2, you may get an error. See this [issue](https://github.com/openai/gym/issues/100) for solution.
* If it doesn't train, it's a good idea to plot reward distribution and record sessions: they may give you some clue. If they don't, call course staff :)
* 20-neuron network is probably not enough, feel free to experiment.

You may find the following snippet useful:

In [ ]:
def visualize_mountain_car(env, agent):
    # Compute policy for all possible x and v (with discretization)
    xs = np.linspace(env.min_position, env.max_position, 100)
    vs = np.linspace(-env.max_speed, env.max_speed, 100)

    grid = np.dstack(np.meshgrid(xs, vs[::-1])).transpose(1, 0, 2)
    grid_flat = grid.reshape(len(xs) * len(vs), 2)
    probs = (
        agent.predict_proba(grid_flat).reshape(len(xs), len(vs), 3).transpose(1, 0, 2)
    )

    # # The above code is equivalent to the following:
    # probs = np.empty((len(vs), len(xs), 3))
    # for i, v in enumerate(vs[::-1]):
    #     for j, x in enumerate(xs):
    #         probs[i, j, :] = agent.predict_proba([[x, v]])[0]

    # Draw policy
    f, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(
        probs,
        extent=(env.min_position, env.max_position, -env.max_speed, env.max_speed),
        aspect="auto",
    )
    ax.set_title("Learned policy: red=left, green=nothing, blue=right")
    ax.set_xlabel("position (x)")
    ax.set_ylabel("velocity (v)")

    # Sample a trajectory and draw it
    states, actions, _ = generate_session(env, agent)
    states = np.array(states)
    ax.plot(states[:, 0], states[:, 1], color="white")

    # Draw every 3rd action from the trajectory
    for (x, v), a in zip(states[::3], actions[::3]):
        if a == 0:
            plt.arrow(x, v, -0.1, 0, color="white", head_length=0.02)
        elif a == 2:
            plt.arrow(x, v, 0.1, 0, color="white", head_length=0.02)


with gym.make("MountainCar-v0", render_mode="rgb_arrary").env as env:
    visualize_mountain_car(env, agent)


### Bonus tasks

* __2.3 bonus__ (0.2 pts) Try to find a network architecture and training params that solve __both__ environments above. 

* __2.4 bonus__ (0.3 pts) Solve continuous action space task with `MLPRegressor` or similar.
  * Since your agent only predicts the "expected" action, you will have to add noise to ensure exploration.
  * Choose one of [MountainCarContinuous-v0](https://gymnasium.farama.org/environments/classic_control/mountain_car_continuous/) (90+ pts to solve), [LunarLanderContinuous-v2](https://gymnasium.farama.org/environments/box2d/lunar_lander/) (`env = gym.make("LunarLander-v2", continuous=True)`)(200+ pts to solve)
  * 0.3 points for solving. Slightly less for getting some results below solution threshold. Note that discrete and continuous environments may have slightly different rules, aside from action spaces.